In [ ]:
import sys
from pathlib import Path
# Add parent directory to path to enable imports
parent_dir = Path().absolute().parent
if str(parent_dir) not in sys.path:
	sys.path.insert(0, str(parent_dir))
from astropy.io import fits
from astropy import units as u
from astropy.cosmology import Planck18 as cosmo
from astropy.cosmology import z_at_value
import matplotlib.pyplot as plt
import healpy as hp
import os
from ligo.skymap.io.fits import read_sky_map
from ligo.skymap.moc import uniq2nest, uniq2pixarea
from model import *
from tqdm import tqdm
import h5py
import torch
import numpy as np
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [ ]:
def sample_moc_skymap(map_file):
    """
    Convert UNIQ to sky coordinates and calculte pixel areas.
    Note: Do not include DISTNORM in the output. For inf values in DISTMU, relace with distmean and diststd from metadata.
    """
    
    # read moc skymap and metadata
    moc_map = read_sky_map(map_file, moc=True, distances=True)
    _, meta = read_sky_map(map_file, nest=True)

    # extract distance meta info
    dist_mean = meta.get('distmean', None)
    dist_std = meta.get('diststd', None)

    uniq = moc_map['UNIQ']
    probdensity = moc_map['PROBDENSITY']
    distmu = moc_map['DISTMU']
    distsigma = moc_map['DISTSIGMA']
    # distnorm = moc_map['DISTNORM']  # not used

    # 1) UNIQ -> order, ipix, nside
    order, ipix = uniq2nest(uniq)

    # 2) caculate pixel area
    dA = uniq2pixarea(uniq)
    # 3) calculate pixel probability
    dP = probdensity * dA
    # 4) calculate theta, phi
    xs  = np.zeros_like(ipix, dtype=np.float32)
    ys = np.zeros_like(ipix, dtype=np.float32)
    zs = np.zeros_like(ipix, dtype=np.float32)
    for k in np.unique(order):
        m = (order == k)
        this_ipix  = ipix[m]
        this_nside = 2 ** k
        theta, phi = hp.pix2ang(this_nside, this_ipix, nest=True)
        # ras[m]  = np.degrees(phi)
        # decs[m] = 90.0 - np.degrees(theta)
        xs[m] = np.sin(theta) * np.cos(phi)   # dec,[0, pi]
        ys[m] = np.sin(theta) * np.sin(phi)   # ra,[0, 2pi]
        zs[m] = np.cos(theta)
    
    # 5) return torch tensors
    gw_mocmap = torch.tensor(np.vstack([xs, ys, zs, dA, 100 * dP, distmu, distsigma]), dtype=torch.float32)   # [7, N_pixels], no distnorm

    # 6) process unnormal distance values
    inf_dist_mu = torch.where(torch.isinf(gw_mocmap[5]))[0]
    gw_mocmap[5, inf_dist_mu] = dist_mean  # set inf to mean value
    gw_mocmap[6, inf_dist_mu] = dist_std   # set inf to std value
    gw_mocmap[5,:] = gw_mocmap[5,:] / 1000.0  # scale down
    gw_mocmap[6,:] = gw_mocmap[6,:] / 1000.0 # scale down

    return gw_mocmap  # [7, N_pixels]

BAND_MAP = {'LSST-u': 0, 'LSST-g': 1, 'LSST-r': 2, 'LSST-i': 3, 'LSST-z': 4, 'LSST-Y': 5}
NUM_BANDS = 6
MAX_LC_LENGTH = 200  # Maximum length of light curves all band
def parse_snana_fits(fits_dir, event_name):
    """
    Parses {event_id}_HEAD.fits and {event_id}_PHOT.fits.
    Extracts multiple light curve realizations for a single GW event.
    
    Args:
        event_id: String ID of the event.
        sim_dir: Directory containing FITS files.
        
    Returns:
        List of tuples: [(values, masks, times), ...]
        Returns empty list if files are missing.
    """

    head_path = os.path.join(fits_dir, f"{event_name}",f"{event_name}_HEAD.FITS")
    phot_path = os.path.join(fits_dir, f"{event_name}",f"{event_name}_PHOT.FITS")

    if not os.path.exists(head_path) or not os.path.exists(phot_path):
        print(f"Warning: FITS files not found for {event_name}")
        return []

    try:
        # open readme file and get MJD explode value
        with open(os.path.join(fits_dir, f"{event_name}",f"{event_name}.README")) as f:
            readme_lines = f.readlines()
            mjd_explode = readme_lines[27].split(":")[1].split()[0]
            mjd_explode = float(mjd_explode)
            print(f"Event {event_name}: MJD explode = {mjd_explode}")
        # Open FITS files
        with fits.open(head_path) as hdul_head, fits.open(phot_path) as hdul_phot:
            # Usually data is in extension 1
            data_head = hdul_head[1].data
            data_phot = hdul_phot[1].data
            
            # Use columns directly (Astropy FITS columns are case-insensitive usually)
            # HEAD columns
            ptrobs_min = data_head['PTROBS_MIN']
            ptrobs_max = data_head['PTROBS_MAX']
            
            # PHOT columns
            mjd_all = data_phot['MJD']
            flux_all = data_phot['FLUXCAL']
            fluxerr_all = data_phot['FLUXCALERR'] # Optional usage
            flt_all = data_phot['BAND'] # Filters

            extracted_lcs = []
            
            # Iterate over each realization in HEAD
            for i in range(len(data_head)):
                # SNANA uses 1-based indexing for pointers, Python uses 0-based
                # Start index: value - 1
                # End index: value (exclusive in python slicing)
                start_idx = ptrobs_min[i] - 1
                end_idx = ptrobs_max[i]

                # get coordinates
                ra = data_head['RA'][i]
                dec = data_head['DEC'][i]
                coordinates = np.array([ra, dec], dtype=np.float32)
                
                # Slicing the PHOT data
                lc_mjd = mjd_all[start_idx : end_idx]
                lc_flux = flux_all[start_idx : end_idx]
                lc_fluxerr = fluxerr_all[start_idx : end_idx]
                lc_flt = flt_all[start_idx : end_idx]

                # Normalization
                std = np.std(lc_flux)
                mean = np.mean(lc_flux)
                lc_flux = (lc_flux - mean) / (std + 1e-8)
                lc_fluxerr = lc_fluxerr / (std + 1e-8)
                
                # --- Format Conversion (to Tensor-ready numpy) ---
                val_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)    # Values matrix (flux)
                err_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)    # Errors matrix (flux errors)
                mask_mat = np.zeros((MAX_LC_LENGTH, NUM_BANDS), dtype=np.float32)
                time_vec = np.zeros((MAX_LC_LENGTH,), dtype=np.float32)
                
                # 1. Time Normalization (Relative to BNS merger time)
                if len(lc_mjd) > 0:
                    rel_times = (lc_mjd - mjd_explode) / 100  # Scale down to manageable range[-0.3, 0.6]
                else:
                    continue # Skip empty light curves

                # 2. Fill Matrices
                # Truncate if longer than MAX_LC_LENGTH
                seq_len = min(len(lc_mjd), MAX_LC_LENGTH)
                if len(lc_mjd) > MAX_LC_LENGTH:
                    print(f"Warning: Light curve for event {event_name} exceeds MAX_LC_LENGTH. Truncating.")
                    # Keep the MAX_LC_LENGTH points with smallest absolute rel_times
                    sorted_indices = np.argsort(np.abs(rel_times))[:MAX_LC_LENGTH]
                    sorted_indices = np.sort(sorted_indices)  # Sort back to chronological order
                    lc_mjd = lc_mjd[sorted_indices]
                    lc_flux = lc_flux[sorted_indices]
                    lc_fluxerr = lc_fluxerr[sorted_indices]
                    lc_flt = lc_flt[sorted_indices]
                    rel_times = rel_times[sorted_indices]
                
                for t in range(seq_len):
                    band_char = lc_flt[t].strip() # Remove whitespace
                    if band_char in BAND_MAP:
                        b_idx = BAND_MAP[band_char]
                        
                        val_mat[t, b_idx] = lc_flux[t]
                        err_mat[t, b_idx] = lc_fluxerr[t]
                        mask_mat[t, b_idx] = 1.0
                        time_vec[t] = rel_times[t]
                
                extracted_lcs.append((val_mat, err_mat, mask_mat, time_vec, coordinates))
                
            return extracted_lcs

    except Exception as e:
        print(f"Error processing FITS for {event_name}: {e}")
        return []

# get data of GW170817A/AT2017gfo

In [ ]:
_BASE = os.environ.get('BASE_DIR', '/fred/oz016/bgao_kn')

gw_params = h5py.File(f'{_BASE}/data/GW170817A/GW170817_GWTC-1.hdf5','r')
highspin_postpiror = gw_params['/IMRPhenomPv2NRT_highSpin_posterior'][:]
lowspin_postpiror = gw_params['/IMRPhenomPv2NRT_lowSpin_posterior'][:]

In [ ]:
spin1 = lowspin_postpiror['spin1']
spin2 = lowspin_postpiror['spin2']
mass1_detector = lowspin_postpiror['m1_detector_frame_Msun']
mass2_detector = lowspin_postpiror['m2_detector_frame_Msun']
distances = lowspin_postpiror['luminosity_distance_Mpc']
cos_inclination = lowspin_postpiror['costheta_jn']
inclination = np.arccos(cos_inclination)

redshift = np.array([z_at_value(cosmo.luminosity_distance, d * u.Mpc) for d in distances])
print(f"Redshift: {np.mean(redshift)}")
print(f"Spin1z: {np.mean(spin1)}, Spin2z: {np.mean(spin2)}")
print(f"Mass1 detector frame: {np.mean(mass1_detector)}, Mass2 detector frame: {np.mean(mass2_detector)}")
print(f"Luminosity distance (Mpc): {np.mean(distances)}")
print(f"Inclination (radians): {np.arccos(np.mean(cos_inclination))}")

In [ ]:
_, meta = read_sky_map(f"{_BASE}/data/GW170817A/bayestar_no_virgo.fits", nest=True)
dist_mean = meta.get('distmean', None)
dist_std = meta.get('diststd', None)
print(f"Distance mean from skymap meta: {dist_mean}, std: {dist_std}")

In [ ]:
pos_gw_scalar = torch.tensor([np.mean(mass1_detector), 
                           np.mean(mass2_detector), 
                           np.mean(spin1), np.mean(spin2), 
                           np.mean(inclination), 
                           dist_mean/100, dist_std/100],dtype=torch.float32) #['mass1', 'mass2', 'spin1z', 'spin2z', 'inclination', 'distmean', 'diststd']
print(f"GW scalars: {pos_gw_scalar}")

In [ ]:
mocmap = read_sky_map(f"{_BASE}/data/GW170817A/bayestar_no_virgo.fits", moc=True, distances=True)
mocmap[:5]

In [ ]:
pos_gw_moc = sample_moc_skymap(f"{_BASE}/data/GW170817A/bayestar_no_virgo.fits")
print(f"GW MOC map shape: {pos_gw_moc.shape}")

## Get optical data

In [ ]:
inclination = np.arccos(np.mean(cos_inclination))
print(f"Inclination (rad): {inclination}")
if inclination > np.pi / 2:
    theta = np.pi - inclination
else:
    theta = inclination
print(f"Viewing angle theta (rad): {theta}")
cos_theta = np.cos(theta)
print(f"Cosine viewing angle cos(theta): {cos_theta}")

In [ ]:
pos_lc = parse_snana_fits(fits_dir=f"{_BASE}/SNANA/SNDATA_ROOT/SIM", event_name="LSST_KN_GW170817_0.1")
pos_val_obs, pos_err_obs, pos_mask_obs, pos_time_obs, pos_coords = pos_lc[0]
pos_opt_v = torch.tensor(pos_val_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
pos_opt_err = torch.tensor(pos_err_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
pos_opt_mask = torch.tensor(pos_mask_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
pos_opt_t = torch.tensor(pos_time_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
pos_opt_coords = torch.tensor(pos_coords, dtype=torch.float32).unsqueeze(0)  # add batch dimension
print(f"Extracted light curve values shape: {pos_opt_v.shape}")

In [ ]:
# load data for test
import matplotlib.pyplot as plt
# 打开PHOT.fits文件
neg_fits=f"{_BASE}/SNANA/SNDATA_ROOT/SIM/LSST_KN_BNS_16/LSST_KN_BNS_16_PHOT.FITS"
pos_fits=f"{_BASE}/SNANA/SNDATA_ROOT/SIM/LSST_KN_GW170817_0.1/LSST_KN_GW170817_0.1_PHOT.FITS"
phot_hdul = fits.open(pos_fits)
phot_data = phot_hdul[1].data

mjd_explode = 63000
# plot light curve for verification
bands = ['LSST-u','LSST-g','LSST-r','LSST-i','LSST-z','LSST-Y']
plt.figure(figsize=(8,6))
plt.axvline(x=mjd_explode, color='r', linestyle='--', label="Merger MJD")
source_data = phot_data[:77]  # FITS索引通常从1开始
# source_data = source_data[np.logical_and(source_data['MJD']>=mjd_explode[0]-5, source_data['MJD']<=mjd_explode[0]+15)]
mjd = source_data['MJD']
flux = source_data['FLUXCAL']
fluxerr = source_data['FLUXCALERR']
# mag, magerr = convert_fluxcal_to_mag(flux=flux.copy(), err=fluxerr)
for band in bands:
    band_mask = source_data['BAND'] == band
    # plt.errorbar(mjd[band_mask], mag[band_mask], yerr=magerr[band_mask], fmt='o', label=band)
    plt.errorbar(mjd[band_mask], flux[band_mask], yerr=fluxerr[band_mask], fmt='o', label=band)
    # plt.errorbar(mjd, flux, yerr=err, fmt='o', label=f'Source {i+1}')
    plt.xlabel('MJD', fontsize=18)
    plt.xticks(fontsize=14)
    plt.ylabel('FLUXCAL', fontsize=18)
    plt.yticks(fontsize=14)
    plt.title('Light Curves from SNANA simulation', fontsize=18)
    plt.legend(fontsize=12)

## generate some negative sample pairs

In [ ]:
neg_mass1, neg_mass2 = 1.8, 1.0
neg_spin1z, neg_spin2z = 0.5, 0.0
neg_inclination = 1.0
neg_distmean, neg_diststd = 500.0, 100.0 
neg_gw_scalars_dicts = {
    'mass1_detector': neg_mass1,
    'mass2_detector': neg_mass2,
    'spin1z': neg_spin1z,
    'spin2z': neg_spin2z,
    'inclination': neg_inclination,
    'distmean': neg_distmean / 1000,
    'diststd': neg_diststd / 1000
}
fully_neg_gw_scalar = torch.tensor(
    [neg_mass1, neg_mass2, neg_spin1z, neg_spin2z, neg_inclination, neg_distmean / 1000, neg_diststd / 1000],
    dtype=torch.float32
)


In [ ]:
neg_gw_moc = sample_moc_skymap(f"{_BASE}/data/bns_skymap/26.fits")
print(f"Negative GW MOC map shape: {neg_gw_moc.shape}")


In [ ]:
neg_lc = parse_snana_fits(fits_dir=f"{_BASE}/SNANA/SNDATA_ROOT/SIM", event_name="LSST_KN_BNS_16")
neg_val_obs, neg_err_obs, neg_mask_obs, neg_time_obs, neg_coords = neg_lc[0]
neg_opt_v = torch.tensor(neg_val_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
neg_opt_err = torch.tensor(neg_err_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
neg_opt_mask = torch.tensor(neg_mask_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
neg_opt_t = torch.tensor(neg_time_obs, dtype=torch.float32).unsqueeze(0)  # add batch dimension
neg_opt_coords = torch.tensor(neg_coords, dtype=torch.float32).unsqueeze(0)  # add batch dimension


In [ ]:
cases = []
cases.append({
    "name": "pos_all",
    "gw_scalar": pos_gw_scalar,
    "gw_moc": pos_gw_moc,
    "opt": "pos"
})

for i, (key, value) in enumerate(neg_gw_scalars_dicts.items()):
    gw_scalar = pos_gw_scalar.clone()
    gw_scalar[i] = value
    cases.append({
        "name": f"change_{key}",
        "gw_scalar": gw_scalar,
        "gw_moc": pos_gw_moc,
        "opt": "pos"
    })

cases.append({
    "name": "neg_skymap_only",
    "gw_scalar": pos_gw_scalar,
    "gw_moc": neg_gw_moc,
    "opt": "pos"
})

cases.append({
    "name": "neg_opt_only",
    "gw_scalar": pos_gw_scalar,
    "gw_moc": pos_gw_moc,
    "opt": "neg"
})

cases.append({
    "name": "neg_all",
    "gw_scalar": fully_neg_gw_scalar,
    "gw_moc": neg_gw_moc,
    "opt": "neg"
})

case_names = [c["name"] for c in cases]

gw_scalars = torch.stack([c["gw_scalar"] for c in cases], dim=0)
gw_moc = torch.stack([c["gw_moc"] for c in cases], dim=0)

def _opt_for_case(case):
    if case["opt"] == "pos":
        return pos_opt_v, pos_opt_err, pos_opt_mask, pos_opt_t, pos_opt_coords
    return neg_opt_v, neg_opt_err, neg_opt_mask, neg_opt_t, neg_opt_coords

opt_v = torch.cat([_opt_for_case(c)[0] for c in cases], dim=0)
opt_err = torch.cat([_opt_for_case(c)[1] for c in cases], dim=0)
opt_mask = torch.cat([_opt_for_case(c)[2] for c in cases], dim=0)
opt_t = torch.cat([_opt_for_case(c)[3] for c in cases], dim=0)
opt_coords = torch.cat([_opt_for_case(c)[4] for c in cases], dim=0)

print(f"Cases: {case_names}")
print(f"GW scalars shape: {gw_scalars.shape}")
print(f"GW MOC map shape: {gw_moc.shape}")
print(f"Optical data shape after repeat: {opt_v.shape}")


Data explanation:
- pos,
- neg-m1,
- meg-m2,
- neg-s1,
- neg-s2,
- neg-cosinclination,
- neg-distmean,
- neg-diststd,
- neg-opt,
- neg-map,
- neg-full-gw

## Inference

In [ ]:
import json
import torch.nn.functional as F
sys.path.insert(0, str(Path().absolute().parent))
from ALBEF_train import compute_credible_level

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG_PATH = f"{_BASE}/ML+GW+KN/Model/args/ALBEF_supcon_v2.json"
CKPT_PATH = f"{_BASE}/data/model/checkpoints/supcon_v2/ALBEF/albef_best.pth"

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

# --- 1. Load Model ---
BATCH_SIZE = gw_scalars.shape[0]
print(f"Loading ALBEF supcon model from {CKPT_PATH}...")
model = GWOpticalALBEFModel(
    gw_scalar_dim=7,
    gw_skymap_channels=7,
    optical_input_dim=6,
    ref_time_dim=cfg.get("ref_dim", 64),
    enc_dim=cfg.get("enc_dim", 128),
    proj_dim=cfg.get("proj_dim", 256),
    fusion_attn_dim=cfg.get("fusion_attn_dim"),
    fusion_hidden_dim=cfg.get("fusion_hidden_dim"),
    temp_init=cfg.get("temp_init", 0.07),
    temp_min=cfg.get("temp_min", 0.01),
    temp_max=cfg.get("temp_max", 100.0),
    fusion_dropout=cfg.get("fusion_dropout", 0.1),
    gw_dropout=cfg.get("gw_dropout", 0.1),
    opt_dropout=cfg.get("opt_dropout", 0.1),
    proj_dropout=cfg.get("proj_dropout", 0.0),
    feature_dropout=cfg.get("feature_dropout", 0.0),
    label_smoothing=cfg.get("label_smoothing", 0.0),
    itc_label_smoothing=cfg.get("itc_label_smoothing", 0.0),
    use_lightweight_gw=cfg.get("use_lightweight_gw", False),
    dual_fusion=cfg.get("dual_fusion", True)
).to(DEVICE)

# Load weights
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
state_dict = checkpoint.get("model_state_dict", checkpoint)
# Strip _orig_mod. prefix from torch.compile'd checkpoints
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print(f"Missing keys: {missing}")
if unexpected:
    print(f"Unexpected keys: {unexpected}")
print(f"Loaded checkpoint from epoch {checkpoint.get('epoch', '?')}, loss={checkpoint.get('loss', '?')}")

model.eval()

# --- 2. Prepare Data ---
gw_s = gw_scalars.to(DEVICE)
gw_m = gw_moc.to(DEVICE)
ov = opt_v.to(DEVICE)
oe = opt_err.to(DEVICE)
om = opt_mask.to(DEVICE)
ot = opt_t.to(DEVICE)
oc = opt_coords.to(DEVICE)

# Create reference time query
N_ref = int(cfg.get("n_ref", 64))
ref_start = float(cfg.get("ref_start", -0.3))
ref_end = float(cfg.get("ref_end", 0.6))
opt_ref_t = torch.linspace(ref_start, ref_end, N_ref, dtype=torch.float32).unsqueeze(0).repeat(BATCH_SIZE, 1).to(DEVICE)

# --- 3. Forward Pass ---
with torch.no_grad():
    # Encode: returns (g, z_l, h_l, H_gw)
    g, z_l, h_l, H_gw = model.encode(gw_s, gw_m, oc, ot, ov, opt_ref_t, om, oe)

    # Compute credible level for dual fusion
    cred_level = compute_credible_level(gw_m, oc)

    # Classification logits
    logits = model.fusion_logits(g, h_l, z_l=z_l, H_gw=H_gw, cred_level=cred_level)
    probs = torch.softmax(logits, dim=1)
    match_probs = probs[:, 1]

    # Embedding similarity (contrastive space)
    feat_g = F.normalize(model.gw_proj(g), p=2, dim=1, eps=1e-8)
    feat_o = F.normalize(model.opt_proj(z_l), p=2, dim=1, eps=1e-8)
    cosine_sim = (feat_g * feat_o).sum(dim=1)  # per-pair cosine similarity

print("=" * 70)
print(f"{'Case':<28s} {'Match Prob':>10s} {'Cosine Sim':>10s} {'Cred Level':>10s}")
print("-" * 70)
for name, prob, sim, cred in zip(case_names, match_probs.tolist(), cosine_sim.tolist(), cred_level.squeeze().tolist()):
    print(f"{name:<28s} {prob:>10.6f} {sim:>10.6f} {cred:>10.6f}")
print("=" * 70)